# Notebook 6 — SciPy: Statistics, Curve Fitting, Signal Processing, and ODEs

Welcome to Notebook 6. SciPy (pronounced "sigh-pie") is the scientific
computing powerhouse that sits on top of NumPy. Where NumPy gives you arrays
and fast maths, SciPy gives you the specialist tools scientists actually reach
for: fitting curves to data, filtering noisy signals, solving differential
equations, and running statistical tests.

By the time you finish this notebook you will have touched the same machinery
that neuroscientists use every day in the lab.

**Four items in this notebook:**

1. Statistical analysis with `scipy.stats`
2. Curve fitting with `scipy.optimize`
3. Signal processing with `scipy.signal` and `scipy.fft`
4. Integration and ODEs with `scipy.integrate`

Each item follows the same structure: concept, worked examples, going deeper,
common confusions, exercises, and a recap. Sections marked *Going deeper* are
optional on a first pass.


---
# Item 1 — Statistical Analysis with `scipy.stats`

> **By the end of this you will be able to:** compute descriptive statistics
> (mean, variance, skewness, kurtosis) on a dataset; evaluate the normal
> distribution (PDF, CDF, inverse CDF); run a t-test to compare two groups;
> and interpret a p-value in plain English. Everything below the core sections
> is optional on a first pass.


## What `scipy.stats` gives you

`scipy.stats` is a toolbox with three shelves:

1. **Probability distributions** — mathematical recipes describing how often
   different values occur (the bell curve, the Poisson distribution for counts,
   and dozens more).
2. **Descriptive statistics** — one-number summaries of a dataset (average,
   spread, shape).
3. **Hypothesis tests** — formal procedures for deciding whether a difference
   you see in data is real or just noise.

Every time a researcher asks "is this drug actually doing something, or is this
just luck?", they are running a hypothesis test from a toolbox exactly like this.


## Descriptive statistics: summarising a dataset in a handful of numbers

### Everyday picture 1: the school report

Imagine 30 students sit an exam. You could list all 30 scores, but a glance
at "class average 72 out of 100, range 45-95" tells you far more, faster.
Descriptive statistics are exactly this: a compact, honest summary of a whole
dataset.

### Everyday picture 2: the weather summary

A weather app does not show you the temperature every second of last month. It
shows mean (average), range, and sometimes "unusually warm" (deviation from
normal). Four numbers replace thousands of readings.

The four key summary values:

- **Mean** -- the arithmetic average. The balancing point of the data.
- **Variance** -- the average of the squared distances from the mean. How
  *spread out* the data are. (Squared so that above-average and below-average
  distances don't cancel each other.)
- **Skewness** -- whether the data leans to one side. A symmetric bell curve
  has skewness zero. A long tail to the right gives positive skew (think
  income distributions: most people earn near the median but a few earn vastly
  more).
- **Kurtosis** -- how "peaky" or "fat-tailed" the distribution is compared
  with a normal bell curve. High kurtosis means more extreme outliers than
  expected.

In neuroscience, you might describe a neuron's (brain cell's) firing rate
(how many spikes per second it produces) across many trials: mean rate, how
variable it is (variance), whether the distribution is symmetric (skewness).


In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Simulate firing rates (spikes per second) for 200 trials of a neuron.
# A neuron is a brain cell; its firing rate is how many electrical pulses
# (action potentials, or "spikes") it sends per second.
rng = np.random.default_rng(42)
firing_rates = rng.normal(loc=20.0, scale=5.0, size=200)
# loc = mean firing rate (20 Hz), scale = standard deviation (5 Hz)
# Hz = Hertz = events per second

# scipy.stats.describe() returns a DescribeResult with all key summaries
desc = stats.describe(firing_rates)

print("Number of observations:", desc.nobs)
print(f"Min: {desc.minmax[0]:.2f}  Max: {desc.minmax[1]:.2f}")
print(f"Mean firing rate:  {desc.mean:.2f} Hz")
print(f"Variance:          {desc.variance:.2f}")
print(f"Skewness:          {desc.skewness:.3f}  (0 = perfectly symmetric)")
print(f"Kurtosis (excess): {desc.kurtosis:.3f}  (0 = normal-shaped tails)")

# Plot a histogram to see the shape we just summarised
plt.figure(figsize=(7, 4))
plt.hist(firing_rates, bins=25, color="steelblue", edgecolor="white", alpha=0.8)
plt.axvline(desc.mean, color="red", linestyle="--", label=f"Mean = {desc.mean:.1f}")
plt.xlabel("Firing rate (Hz)")
plt.ylabel("Number of trials")
plt.title("Simulated neuron firing rates across 200 trials")
plt.legend()
plt.tight_layout()
plt.show()


## The normal distribution: the bell curve in precise mathematical form

### Everyday picture 1: patient heights

Measure the heights of 10 000 adults. Most cluster near the average (say 170 cm),
fewer are very short or very tall, and the pattern is almost perfectly symmetric.
That shape is the normal distribution, the "bell curve."

### Everyday picture 2: measurement noise

You weigh the same tablet 50 times on a precise scale. You get 50 slightly
different readings, not because the tablet changes but because every measurement
has tiny random errors. Those errors pile up into a bell curve centred on the
true weight.

### The PDF and CDF in plain English

**PDF (probability density function):** The height of the bell curve at any
point. A higher curve means values there are more common. It is NOT a
probability by itself -- you have to look at an area (a range of values) to
get a probability. Think of it as "how crowded is this part of the number line?"

**CDF (cumulative distribution function):** "What fraction of values fall
BELOW this point?" Start at the far left (zero) and sweep right; the CDF rises
from 0 to 1. At the mean, the CDF equals 0.5 (half the values are below average).

**PPF (percent-point function, also called the inverse CDF or quantile function):**
The reverse of the CDF. "What value has exactly p% of the data below it?" Feed
in 0.95 and it tells you the 95th percentile.

In neuroscience: if you have a neuron's typical firing distribution you can ask
"what fraction of trials had firing rates above 30 Hz?" -- that is a CDF lookup.


In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# scipy.stats.norm represents the normal (Gaussian) distribution.
# Parameters: loc = mean, scale = standard deviation.
mu, sigma = 20.0, 5.0   # mean 20 Hz, SD 5 Hz

x = np.linspace(0, 45, 400)  # a range of firing-rate values to evaluate

# --- PDF ---
pdf_values = stats.norm.pdf(x, loc=mu, scale=sigma)
# Each value: "how dense is probability here?" High in the middle, near zero at edges.

# --- CDF ---
cdf_values = stats.norm.cdf(x, loc=mu, scale=sigma)
# Each value: "what fraction of neurons fire SLOWER than this rate?"

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x, pdf_values, color="steelblue", linewidth=2)
axes[0].fill_between(x, pdf_values, where=(x >= 30), alpha=0.4, color="red",
                     label="P(rate > 30 Hz)")
axes[0].set_xlabel("Firing rate (Hz)")
axes[0].set_ylabel("Probability density")
axes[0].set_title("PDF: the bell curve shape")
axes[0].legend()

axes[1].plot(x, cdf_values, color="darkorange", linewidth=2)
axes[1].axhline(0.5, linestyle="--", color="gray", label="50th percentile (mean)")
axes[1].axvline(mu, linestyle="--", color="gray")
axes[1].set_xlabel("Firing rate (Hz)")
axes[1].set_ylabel("Cumulative probability")
axes[1].set_title("CDF: fraction of neurons firing slower than x")
axes[1].legend()

plt.tight_layout()
plt.show()

# Numerical answers
p_above_30 = 1 - stats.norm.cdf(30, loc=mu, scale=sigma)
print(f"P(firing rate > 30 Hz) = {p_above_30:.3f}  ({p_above_30*100:.1f}%)")

# PPF: what firing rate marks the top 5% of active neurons?
top_5_percent_threshold = stats.norm.ppf(0.95, loc=mu, scale=sigma)
print(f"95th percentile of firing rate = {top_5_percent_threshold:.2f} Hz")

# Z-score: how many standard deviations is a voltage reading from the mean?
# Useful for detecting unusually large signals.
voltage_readings = np.array([-68.0, -72.5, -55.0, -70.1, -48.0])
# Membrane potential (in millivolts, mV): the electrical voltage across a
# neuron's membrane. Normal resting value is around -70 mV.
z_scores = stats.zscore(voltage_readings)
print("\nVoltage readings (mV):", voltage_readings)
print("Z-scores:            ", np.round(z_scores, 2))
print("(Z > 2 or < -2 means unusually far from the group average)")


## The t-test: did the treatment actually work?

### Everyday picture 1: comparing two drugs

A clinical trial gives Drug A to 40 patients and a placebo (sugar pill) to
40 others and measures blood pressure reduction. The Drug A group average is
12 mmHg; the placebo average is 9 mmHg. Is that 3-point difference real, or
could it arise just from random variation in who happened to be in each group?
That is exactly what a t-test answers.

### Everyday picture 2: before vs after

A single group of patients takes a medication. You measure their symptoms before
and after. Did the score change beyond what random day-to-day fluctuation would
produce? A paired t-test (here we use the one-sample version) tells you.

### The p-value in plain English

The t-test produces a **p-value** (p for probability). It answers this specific
question:

> "IF the treatment had absolutely zero effect, how likely would we be to see
> a difference this large (or larger) just by chance?"

So a p-value of 0.03 means: "If the drug did nothing, there is only a 3%
chance we would see a gap this big between groups by random luck."

Convention: if p < 0.05 (less than 5% chance the result is pure luck), we
call the result **statistically significant** and take it as evidence the
treatment did something real. This threshold is arbitrary but widely used.

**Important:** a small p-value tells you the difference is unlikely to be
noise. It does NOT tell you the difference is large enough to matter clinically.
A tiny but real effect can be significant with enough data.

In neuroscience: do neurons fire more during a visual stimulus than in the
baseline period? A t-test on the firing rates from each condition answers this.


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(7)

# Simulate firing rates (Hz) for a neuron during two experimental conditions.
# Condition A: baseline (no stimulus) -- mean 15 Hz
# Condition B: visual stimulus active -- mean 20 Hz
# The neuron fires faster during the stimulus; is this statistically real?

rates_baseline = rng.normal(loc=15.0, scale=4.0, size=30)   # 30 trials
rates_stimulus = rng.normal(loc=20.0, scale=4.5, size=30)   # 30 trials

# Independent-samples t-test (two groups, different trials)
t_stat, p_value = stats.ttest_ind(rates_baseline, rates_stimulus)
# t_stat: how many "standard errors" apart the two means are
# p_value: probability of a gap this large if the two conditions were identical

print("=== Independent t-test: baseline vs stimulus ===")
print(f"Baseline mean:  {rates_baseline.mean():.2f} Hz")
print(f"Stimulus mean:  {rates_stimulus.mean():.2f} Hz")
print(f"Difference:     {rates_stimulus.mean() - rates_baseline.mean():.2f} Hz")
print(f"t statistic:    {t_stat:.3f}")
print(f"p-value:        {p_value:.4f}")
if p_value < 0.05:
    print("Conclusion: difference is statistically significant (p < 0.05).")
    print("The stimulus appears to genuinely change this neuron's firing rate.")
else:
    print("Conclusion: not significant. Could be random variation.")

print()

# One-sample t-test: is our neuron's baseline rate different from the
# lab standard of 12 Hz that we expected?
t_one, p_one = stats.ttest_1samp(rates_baseline, popmean=12.0)
print("=== One-sample t-test: baseline vs expected 12 Hz ===")
print(f"Sample mean: {rates_baseline.mean():.2f} Hz  vs  expected 12.0 Hz")
print(f"t = {t_one:.3f},  p = {p_one:.4f}")
if p_one < 0.05:
    print("Baseline rate is significantly different from 12 Hz.")
else:
    print("No significant difference from 12 Hz.")


---
> ## Going deeper (optional on a first pass)
>
> **Shapiro-Wilk test for normality.** The t-test assumes your data are roughly
> normally distributed. The Shapiro-Wilk test checks this assumption.
> `stats.shapiro(data)` returns a p-value; if p < 0.05, the data deviate
> significantly from a normal shape and you should consider alternatives.
>
> **Mann-Whitney U test.** A non-parametric alternative to the independent
> t-test that does not assume normality. It compares the *ranks* of values
> rather than the values themselves. Use `stats.mannwhitneyu(a, b)`.
>
> **Bonferroni correction.** If you run 20 t-tests at once (one per brain
> region, say) and use p < 0.05 for each, you expect one false positive by
> pure chance (5% of 20 = 1). The Bonferroni correction divides the threshold
> by the number of tests: significance requires p < 0.05 / 20 = 0.0025.
> This is conservative but straightforward.
>
> **Effect size.** Cohen's d measures *how big* the difference is, independent
> of sample size. `d = (mean1 - mean2) / pooled_std`. Values: 0.2 = small,
> 0.5 = medium, 0.8 = large. A statistically significant result with d = 0.1
> may be practically irrelevant.


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(99)

# Going deeper: Shapiro-Wilk + Mann-Whitney + Bonferroni demo

rates_a = rng.normal(20, 5, 40)
rates_b = rng.exponential(scale=20, size=40)  # non-normal (skewed) distribution

# 1. Check normality
stat_sw_a, p_sw_a = stats.shapiro(rates_a)
stat_sw_b, p_sw_b = stats.shapiro(rates_b)
print(f"Shapiro-Wilk p-value, group A: {p_sw_a:.3f}  (>0.05 = looks normal)")
print(f"Shapiro-Wilk p-value, group B: {p_sw_b:.3f}  (>0.05 = looks normal)")

# 2. Non-parametric test (safe for non-normal data)
u_stat, p_mw = stats.mannwhitneyu(rates_a, rates_b, alternative="two-sided")
print(f"\nMann-Whitney U p-value: {p_mw:.4f}")

# 3. Bonferroni: if we ran 10 tests, our adjusted threshold
n_tests = 10
bonferroni_threshold = 0.05 / n_tests
print(f"\nBonferroni threshold for {n_tests} tests: {bonferroni_threshold}")
print(f"Our p = {p_mw:.4f} vs threshold {bonferroni_threshold:.4f}")


## Common questions and confusions

**"What does p < 0.05 actually mean?"** It means: if the null hypothesis (no
effect) were true, there would be less than a 5% chance of getting a result
as extreme as ours. It does NOT mean there is a 95% probability the effect is
real, though many people misread it that way.

**"Why do I square the deviations for variance instead of just averaging them?"**
If you average the raw deviations (distances from the mean), the positives and
negatives cancel and you get zero every time. Squaring makes all deviations
positive so they accumulate rather than cancel.

**"Is the CDF value at x = 20 the probability that a value equals exactly 20?"**
No. For a continuous distribution, the probability of any exact value is zero
(infinitely thin slice). The CDF gives the probability of being *at or below*
20. Probabilities for ranges come from differences: P(15 < X < 25) = CDF(25) - CDF(15).

**"When should I use ttest_ind vs ttest_1samp?"** Use ttest_ind when comparing
two separate groups (drug vs placebo). Use ttest_1samp when comparing one group's
mean to a known reference value (my patients vs the population norm).

**"Does a significant p-value prove causation?"** No. It says the difference is
unlikely to be noise. It says nothing about why.


## Your exercises

Predict each answer first, then run and check.

1. Generate 100 values from a normal distribution with mean 50 and SD 10.
   Use `stats.describe()` to print the mean, variance, skewness, and kurtosis.
   Does skewness come out close to 0?

2. Using `stats.norm.cdf()`, compute the probability that a patient's resting
   membrane potential (normally distributed, mean -70 mV, SD 5 mV) falls below
   -80 mV. Is this a common or rare event?

3. Find the 99th percentile of the distribution in exercise 2. Use `stats.norm.ppf()`.

4. Generate two groups: `group_a` with mean 30 Hz (n=25 trials) and `group_b`
   with mean 33 Hz (n=25 trials), both SD = 6 Hz. Run a t-test. Is the
   3 Hz difference statistically significant?

5. Repeat exercise 4 but with n=200 trials per group. Does significance change?
   Why? (Think about what sample size does to our ability to detect real effects.)

6. *(Stretch.)* Generate 50 values from a normal distribution and 50 from an
   exponential distribution. Run `stats.shapiro()` on both. Then run both
   `stats.ttest_ind()` and `stats.mannwhitneyu()`. Compare the p-values.
   When would you trust one over the other?


In [ ]:
# Exercise 1
# your code here

# Exercise 2
# your code here

# Exercise 3
# your code here

# Exercise 4
# your code here

# Exercise 5
# your code here

# Exercise 6 (stretch)
# your code here


## The irreducible core

1. `scipy.stats.describe()` gives mean, variance, skewness, and kurtosis in one call.
2. The **PDF** is the bell-curve height ("how common is this value?"); the **CDF**
   is the running total ("what fraction are below this?").
3. A **p-value** answers: "how likely is this result if there were no real effect?"
   Below 0.05 is the conventional significance threshold.
4. `ttest_ind` compares two separate groups; `ttest_1samp` compares one group
   to a reference value.
5. Statistical significance and practical importance are different things -- always
   consider effect size alongside the p-value.

**You have got it when:** given two arrays of neuron firing rates from two
conditions, you can run a t-test, read off the p-value, and explain in plain
English what it does and does not tell you.


---
# Item 2 — Curve Fitting with `scipy.optimize`

> **By the end of this you will be able to:** define a model function, use
> `curve_fit` to find the best-fit parameters, extract the optimal values and
> their uncertainty, and interpret the quality of the fit with residuals.
> Everything below the core sections is optional on a first pass.


## What curve fitting means

### Everyday picture 1: calibrating a thermometer

You have a new electronic thermometer. You dip it in ice water (0 degrees C) and
boiling water (100 degrees C) and read off voltages. You want to find the formula
that converts voltage to temperature for every value in between. You are "fitting
a curve" (here, a straight line) to data points.

### Everyday picture 2: a dose-response curve in medicine

A pharmacologist gives patients five different doses of a drug and records the
response at each dose. The data follow an S-shaped curve: low doses do nothing,
middle doses ramp up the effect, high doses saturate it. The pharmacologist fits
a mathematical S-curve to find the dose that produces 50% of maximum effect
(called the EC50). That fitting is `curve_fit`.

**The core idea:**

You say: "I believe my data follow this shape, described by this formula with
some unknown constants." `curve_fit` adjusts those constants until the formula
matches your data as closely as possible (it minimises the sum of squared errors
between formula and data).

What it returns:
- **`popt`** (optimal parameters): the best-fit values of your constants.
- **`pcov`** (parameter covariance matrix): a measure of *uncertainty* in each
  parameter. The square root of the diagonal gives the standard errors. If a
  parameter's standard error is larger than the parameter itself, the fit is
  poorly constrained.


## Residuals and goodness of fit

**Residuals** are the differences between your data and the fitted curve at each
point: `residual = data - model(x, *popt)`.

- If the fit is good, residuals are small and scattered randomly around zero
  (no systematic pattern).
- If residuals show a pattern (a bend, a systematic offset), the model shape
  is wrong -- you've picked the wrong mathematical form.

Think of measuring a table with a ruler. The residuals are the tiny gaps between
the ruler markings and the actual edge. Small, random gaps mean a good fit.
A systematic curve means your ruler is bent.


In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------
# Comp-neuro example 1: fit an exponential decay to a membrane voltage trace
# -----------------------------------------------------------------------
# When a neuron receives no input, its membrane potential (voltage across
# the cell membrane) decays exponentially back to its resting value.
# The rate of decay is characterised by the "time constant" tau (Greek letter
# for "t"), usually denoted tau. Larger tau = slower decay.
# Units: tau is in milliseconds (ms), voltage in millivolts (mV).

# True (hidden) parameters we are trying to recover:
V0_true = -50.0    # initial voltage (mV) -- pulled from resting by a brief pulse
V_rest = -70.0     # resting membrane potential (mV)
tau_true = 20.0    # time constant in ms: time for decay to fall to 1/e of initial

t = np.linspace(0, 100, 200)   # time axis, 0 to 100 ms

# Exponential decay formula: V(t) = V_rest + (V0 - V_rest) * exp(-t / tau)
rng = np.random.default_rng(0)
noise = rng.normal(0, 0.8, size=t.size)   # realistic measurement noise
V_data = V_rest + (V0_true - V_rest) * np.exp(-t / tau_true) + noise

# Define the MODEL FUNCTION for curve_fit.
# The first argument MUST be the independent variable (time).
# The remaining arguments are the parameters to fit.
def membrane_decay(t, V0, V_rest, tau):
    return V_rest + (V0 - V_rest) * np.exp(-t / tau)

# Initial guesses for the parameters -- they do not need to be perfect,
# but they should be in the right ballpark (see Going Deeper).
p0 = [-55.0, -68.0, 15.0]

popt, pcov = curve_fit(membrane_decay, t, V_data, p0=p0)
# popt: array of best-fit [V0, V_rest, tau]
# pcov: 3x3 matrix; diagonal entries are variances of each parameter

V0_fit, Vrest_fit, tau_fit = popt
perr = np.sqrt(np.diag(pcov))   # standard errors (uncertainty) of each parameter

print("=== Membrane decay fit ===")
print(f"V0    true: {V0_true:.1f}   fit: {V0_fit:.2f} +/- {perr[0]:.2f} mV")
print(f"V_rest true: {V_rest:.1f}  fit: {Vrest_fit:.2f} +/- {perr[1]:.2f} mV")
print(f"tau   true: {tau_true:.1f}   fit: {tau_fit:.2f} +/- {perr[2]:.2f} ms")

# Residuals: how far is the fit from each data point?
V_fit = membrane_decay(t, *popt)  # *popt unpacks the array as individual arguments
residuals = V_data - V_fit

fig, axes = plt.subplots(2, 1, figsize=(9, 6))
axes[0].plot(t, V_data, "o", color="steelblue", markersize=2, label="Noisy data")
axes[0].plot(t, V_fit, color="red", linewidth=2, label=f"Fit (tau={tau_fit:.1f} ms)")
axes[0].set_xlabel("Time (ms)")
axes[0].set_ylabel("Membrane potential (mV)")
axes[0].set_title("Exponential decay fit to membrane voltage")
axes[0].legend()

axes[1].plot(t, residuals, "o", color="gray", markersize=2)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("Residual (mV)")
axes[1].set_title("Residuals (good fit: scattered randomly around 0, no pattern)")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------
# Comp-neuro example 2: fit a sigmoid to a psychometric curve
# -----------------------------------------------------------------------
# A "psychometric curve" shows how the probability of a correct response
# changes with stimulus strength. At low stimulus levels the subject guesses
# (50% correct); at high levels they always get it right (100% correct).
# This S-shape is described by a sigmoid (logistic) function.

# Stimulus contrast levels (0 to 1 = fraction of maximum contrast)
contrast = np.array([0.0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0])
# Fraction of trials answered correctly at each contrast level
p_correct = np.array([0.50, 0.52, 0.57, 0.65, 0.75, 0.82, 0.89, 0.94, 0.97, 0.99])

# Logistic sigmoid: p(x) = 1 / (1 + exp(-(x - x50) / slope))
# x50 = the contrast at which performance is midway between 50% and 100%
# slope = how steeply the curve rises
def sigmoid(x, x50, slope):
    return 1.0 / (1.0 + np.exp(-(x - x50) / slope))

popt, pcov = curve_fit(sigmoid, contrast, p_correct, p0=[0.3, 0.1])
x50_fit, slope_fit = popt
perr = np.sqrt(np.diag(pcov))

print("=== Psychometric curve fit ===")
print(f"x50  (threshold contrast): {x50_fit:.3f} +/- {perr[0]:.3f}")
print(f"slope:                     {slope_fit:.3f} +/- {perr[1]:.3f}")

x_smooth = np.linspace(0, 1, 300)
plt.figure(figsize=(7, 4))
plt.plot(contrast, p_correct, "o", color="steelblue", label="Data", markersize=7)
plt.plot(x_smooth, sigmoid(x_smooth, *popt), color="red", linewidth=2,
         label=f"Sigmoid fit (threshold={x50_fit:.2f})")
plt.axhline(0.5, linestyle=":", color="gray")
plt.xlabel("Stimulus contrast")
plt.ylabel("P(correct)")
plt.title("Psychometric curve: sigmoid fit to behavioural data")
plt.legend()
plt.tight_layout()
plt.show()


---
> ## Going deeper (optional on a first pass)
>
> **`scipy.optimize.minimize()`.** `curve_fit` is a special-purpose tool for
> fitting a model function to (x, y) data. `minimize()` is the general engine:
> give it any function to minimise and it finds the input that makes it smallest.
> You can minimise negative log-likelihood for probabilistic models, or any
> custom loss function.
>
> **Bounds and constraints.** `curve_fit` accepts a `bounds` argument:
> `bounds=([lower1, lower2], [upper1, upper2])`. This prevents the optimizer
> from trying physically impossible values (negative time constants, for
> instance). Always use bounds if parameters have physical meaning.
>
> **Initial parameter guesses matter enormously.** `curve_fit` uses an
> iterative algorithm (Levenberg-Marquardt by default) that starts at your
> `p0` guess and walks downhill. If the starting point is too far from the
> true answer, it can get stuck in a local minimum (a valley that is not the
> deepest valley). For difficult fits: try multiple random starting points and
> keep the result with the smallest residual sum of squares.
>
> **Gaussian tuning curve.** Many neurons in visual cortex respond to a
> preferred orientation (angle of a bar of light) and less to other angles.
> Plotting response vs orientation gives a bell curve (Gaussian). Fitting a
> Gaussian to this "tuning curve" extracts the preferred orientation and the
> tuning width: `f(x) = A * exp(-(x - mu)**2 / (2*sigma**2))`.


In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# Going deeper: Gaussian fit to a neural tuning curve
# Many neurons respond best to one stimulus orientation and less to others.

orientations = np.array([0, 30, 60, 90, 120, 150, 180])   # degrees
# Firing rate (Hz) at each orientation -- neuron prefers ~90 degrees
responses = np.array([3.0, 8.0, 18.0, 35.0, 20.0, 9.0, 4.0])

def gaussian(x, A, mu, sigma, baseline):
    # A = peak amplitude above baseline, mu = preferred orientation,
    # sigma = tuning width, baseline = spontaneous firing rate
    return baseline + A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

popt, pcov = curve_fit(gaussian, orientations, responses,
                       p0=[30, 90, 30, 3],
                       bounds=([0, 0, 5, 0], [200, 180, 180, 50]))
A_fit, mu_fit, sigma_fit, base_fit = popt
print(f"Preferred orientation: {mu_fit:.1f} degrees")
print(f"Tuning width (sigma):  {sigma_fit:.1f} degrees")
print(f"Peak firing rate:      {base_fit + A_fit:.1f} Hz")

x_smooth = np.linspace(0, 180, 300)
plt.figure(figsize=(7, 4))
plt.plot(orientations, responses, "o", markersize=8, label="Measured responses")
plt.plot(x_smooth, gaussian(x_smooth, *popt), linewidth=2, color="red",
         label=f"Gaussian fit (preferred={mu_fit:.0f} deg)")
plt.xlabel("Stimulus orientation (degrees)")
plt.ylabel("Firing rate (Hz)")
plt.title("Neural tuning curve with Gaussian fit")
plt.legend()
plt.tight_layout()
plt.show()


## Common questions and confusions

**"What is `*popt` doing in `model(x, *popt)`?"** The asterisk unpacks the
array `popt` into individual arguments. If `popt = [1.5, 20.0]` then
`model(x, *popt)` is the same as `model(x, 1.5, 20.0)`. It saves you from
writing `model(x, popt[0], popt[1])`.

**"Why does my fit return absurd parameter values?"** Almost always a bad
initial guess. Try plotting your data first, estimate the parameters by eye,
and use those as `p0`. Also check that your model function is mathematically
correct (test it manually before passing to `curve_fit`).

**"What if `pcov` contains `inf` values?"** This means the fit failed to
converge, or the model is not identifiable (two parameters that cannot be
separated from each other given the data). Simplify the model, add more data,
or tighten bounds.

**"Can I fit non-curve things, like surfaces or models with multiple inputs?"**
Yes. `curve_fit` works for any function where the first argument is the
independent variable (which can be a 2D array for a surface). For very general
optimisation, use `scipy.optimize.minimize`.


## Your exercises

1. Generate noisy data: `t = np.linspace(0, 5, 100)`,
   `y = 3.0 * np.exp(-t / 1.5) + rng.normal(0, 0.2, 100)`.
   Define an exponential decay model and fit it. Print the recovered amplitude
   and time constant with uncertainties.

2. Generate data from a straight line `y = 2.5 * x + 1.0` plus noise.
   Fit a linear model (`def linear(x, m, c): return m*x + c`) using
   `curve_fit`. Compare with `np.polyfit(x, y, 1)`. Do they agree?

3. Create a simple dose-response curve with 8 data points that rises from 0 to
   1 as dose increases. Fit a logistic sigmoid. Report the EC50 (the dose at
   which response = 0.5).

4. Modify the membrane decay worked example to also recover `V_rest` as a free
   parameter (it is already a parameter in the function). Try starting `p0`
   with `V_rest = -60` instead of -70. Does the fit still converge correctly?

5. *(Stretch.)* Generate data from a sum of two exponentials:
   `y = 2*exp(-t/0.5) + 5*exp(-t/5)` plus noise. Try fitting a single
   exponential. Look at the residuals. Then fit a double exponential. Compare
   the residual patterns. What do residuals tell you about model choice?


In [ ]:
# Exercise 1
# your code here

# Exercise 2
# your code here

# Exercise 3
# your code here

# Exercise 4
# your code here

# Exercise 5 (stretch)
# your code here


## The irreducible core

1. `curve_fit(model, xdata, ydata, p0=guess)` adjusts `model`'s parameters
   to minimise the squared difference between the model and the data.
2. `popt` holds the best-fit parameter values; `np.sqrt(np.diag(pcov))` gives
   their standard errors (uncertainties).
3. **Residuals = data - fit**. Random residuals near zero mean a good fit; a
   systematic pattern means the model shape is wrong.
4. Always provide sensible initial guesses `p0`; poor guesses can trap the
   optimizer in the wrong solution.
5. Use `bounds` to constrain parameters to physically meaningful ranges.

**You have got it when:** you can write a three-line model function, pass it
to `curve_fit`, and report the best-fit parameter with its uncertainty.


---
# Item 3 — Signal Processing with `scipy.signal` and `scipy.fft`

> **By the end of this you will be able to:** design and apply a digital filter
> to remove noise from a signal, compute the frequency content of a signal
> using the Fourier transform, detect peaks automatically, and apply these
> to neuroscientific data. Everything below the core sections is optional on
> a first pass.


## What signal processing is

A **signal** is any measurement that changes over time: your EEG, a stock
price, the sound from a microphone, or the electrical activity of a brain
region.

Real signals always contain **noise** (unwanted random variation) mixed in
with the actual information. Signal processing is the set of tools for
separating the information from the noise.

### Everyday picture 1: noise-cancelling headphones

A noise-cancelling headphone listens to the background drone of an aeroplane
engine and suppresses those low frequencies while letting through the frequency
range of speech and music. It is running a filter in real time: block certain
frequency bands, pass others.

### Everyday picture 2: tuning a radio

When you tune a radio you are selecting one frequency (say, 98.5 MHz) and
rejecting all others. The circuit inside is a band-pass filter: it passes a
narrow band of frequencies and blocks everything else.

### Key vocabulary

- **Frequency (Hz = Hertz):** the number of complete oscillation cycles per
  second. A 10 Hz brain rhythm completes 10 cycles per second.
- **Low-pass filter:** passes low frequencies (slow changes), blocks high
  frequencies (fast noise). Smooths the signal.
- **High-pass filter:** passes high frequencies, blocks slow drift.
- **Band-pass filter:** passes a specific frequency range. Used to isolate
  brain rhythms like alpha (8-12 Hz) or gamma (30-80 Hz).
- **LFP (local field potential):** the slow, oscillating electrical signal
  recorded from a brain region, reflecting the summed activity of many neurons
  in the vicinity. Typically 0.1-300 Hz. It is the "hum" of a local brain area.
- **Power spectrum:** a graph showing how much energy (power) is present at
  each frequency. A big spike at 10 Hz means the signal oscillates at 10 Hz.


## Filtering with `butter()` and `sosfilt()`

`scipy.signal.butter()` designs a Butterworth filter (a smooth, popular design)
and returns the filter coefficients.
`scipy.signal.sosfilt()` applies the filter to your data.

Why "SOS"? "Second-Order Sections" -- a numerically stable way to represent
the filter. Just use `output="sos"` in `butter()` and pass the result to
`sosfilt()`. You do not need to understand the maths to use it correctly.


In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------
# Simulated LFP (local field potential): a 10 Hz brain oscillation buried in noise
# LFP = the slow electrical signal from a brain region; measured in microvolts (uV)
# -----------------------------------------------------------------------
rng = np.random.default_rng(5)

fs = 1000.0          # sampling rate: 1000 samples per second (1000 Hz)
duration = 2.0       # seconds
t = np.arange(0, duration, 1.0 / fs)   # time array

# True signal: a 10 Hz oscillation (alpha-band brain rhythm) at 50 uV amplitude
# Alpha oscillations (8-12 Hz) appear during relaxed wakefulness, eyes closed.
clean_signal = 50.0 * np.sin(2 * np.pi * 10 * t)

# Add high-frequency noise (simulating electrical interference, muscle artefacts)
noise = rng.normal(0, 30.0, size=t.size)
lfp = clean_signal + noise    # the raw, noisy LFP you would record

# -----------------------------------------------------------------------
# Design a low-pass Butterworth filter
# Low-pass: keeps frequencies BELOW cutoff, removes high-frequency noise
# -----------------------------------------------------------------------
cutoff_hz = 30.0    # remove everything faster than 30 Hz (keeps our 10 Hz rhythm)
order = 4           # filter order: higher = steeper cutoff, but risk of ringing

# butter() returns SOS (second-order sections) coefficients
sos = signal.butter(order, cutoff_hz, btype="low", fs=fs, output="sos")
# btype="low" = low-pass filter
# fs = sampling frequency in Hz (needed to convert cutoff from Hz to normalised units)

# Apply the filter -- sosfilt runs the filter along the data
lfp_filtered = signal.sosfilt(sos, lfp)

# Plot raw vs filtered
plt.figure(figsize=(11, 5))
plt.plot(t[:500], lfp[:500], color="lightblue", linewidth=0.8, label="Raw LFP (noisy)")
plt.plot(t[:500], lfp_filtered[:500], color="navy", linewidth=2, label="Filtered (low-pass 30 Hz)")
plt.plot(t[:500], clean_signal[:500], color="red", linewidth=1.5, linestyle="--",
         label="True 10 Hz signal")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (uV)")
plt.title("Low-pass filtering recovers the 10 Hz LFP oscillation")
plt.legend()
plt.tight_layout()
plt.show()


## The Fourier transform: decomposing a signal into its frequencies

### Everyday picture 1: a prism splitting white light

White light looks featureless but is actually a mixture of all colours (all
frequencies of light). A glass prism separates them into a rainbow. The Fourier
transform does exactly this for any signal: it decomposes it into the individual
frequencies it is made of, and tells you how strong each frequency is.

### Everyday picture 2: a music equaliser

The graphic equaliser on a stereo shows bars for bass, mid, treble. Each bar
shows how much energy is present at that frequency range. That display IS the
Fourier transform of the audio, computed in real time.

### How `scipy.fft.fft()` works

`scipy.fft.fft(signal)` returns complex numbers -- one per frequency -- whose
magnitude squared is the **power** at that frequency. You do not need to
understand complex numbers; just take `np.abs(fft_output) ** 2` for power.

`scipy.fft.fftfreq(n, d=1/fs)` returns the corresponding frequency for each
element. The output has positive and negative frequencies; you usually only
plot the positive half.

**Power spectrum:** the plot of power vs frequency. Peaks show you which
frequencies dominate the signal.


In [ ]:
import numpy as np
from scipy import fft as scipy_fft
import matplotlib.pyplot as plt

# Reuse the LFP from above (re-create here for standalone running)
rng = np.random.default_rng(5)
fs = 1000.0
duration = 2.0
t = np.arange(0, duration, 1.0 / fs)
clean_signal = 50.0 * np.sin(2 * np.pi * 10 * t)
noise = rng.normal(0, 30.0, size=t.size)
lfp = clean_signal + noise

# -----------------------------------------------------------------------
# Compute the power spectrum using the FFT
# FFT = Fast Fourier Transform: an efficient algorithm to compute the
# Fourier transform of a digital signal.
# -----------------------------------------------------------------------
N = len(lfp)                     # number of samples

# fft() returns complex amplitudes; one per frequency component
fft_vals = scipy_fft.fft(lfp)

# Convert complex amplitudes to power (amplitude squared)
power = (np.abs(fft_vals) ** 2) / N    # normalise by N so power doesn't grow with recording length

# fftfreq gives the frequency for each FFT bin
freqs = scipy_fft.fftfreq(N, d=1.0 / fs)   # d = spacing between samples (seconds)

# Only keep the positive-frequency half (the negative half is a mirror image)
pos_mask = freqs >= 0
freqs_pos = freqs[pos_mask]
power_pos = power[pos_mask]

plt.figure(figsize=(9, 4))
plt.plot(freqs_pos, power_pos, color="steelblue", linewidth=1.2)
plt.axvline(10, color="red", linestyle="--", label="10 Hz (true signal)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power (arbitrary units)")
plt.title("Power spectrum of noisy LFP: spike at 10 Hz reveals the hidden rhythm")
plt.xlim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

# Report the dominant frequency
peak_freq = freqs_pos[np.argmax(power_pos)]
print(f"Dominant frequency in signal: {peak_freq:.1f} Hz  (true: 10 Hz)")


## Detecting peaks with `find_peaks()`

`scipy.signal.find_peaks(signal)` scans a 1D array and returns the indices of
local maxima (peaks). You can set thresholds for height, prominence (how much
a peak sticks up above nearby values), and minimum separation.

In neuroscience: detecting the exact moment of each action potential (spike)
in a voltage recording is called "spike sorting" or "spike detection." This is
one of the most fundamental data-processing steps in electrophysiology.


In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------
# Detect spikes in a simulated voltage trace
# -----------------------------------------------------------------------
# An action potential (spike) is a rapid, stereotyped voltage jump:
# the membrane potential shoots up to about +40 mV and back down in 1-2 ms.
rng = np.random.default_rng(3)
fs = 20000.0   # 20 kHz (typical for spike recordings)
t = np.arange(0, 0.2, 1.0 / fs)   # 200 ms recording

# Baseline: noisy resting potential around -65 mV
voltage = rng.normal(-65.0, 2.0, size=t.size)

# Insert 4 synthetic spikes at known times
spike_times_ms = [20, 60, 110, 170]   # milliseconds
for st_ms in spike_times_ms:
    idx = int(st_ms * 1e-3 * fs)      # convert ms to sample index
    if idx + 20 < len(voltage):
        # Simple spike shape: sharp rise then fall
        spike_shape = np.array([0, 10, 40, 15, -20, -10, -5, 0, 2, 1]) * 1.0
        end = min(idx + len(spike_shape), len(voltage))
        voltage[idx:end] += spike_shape[:end - idx]

# Detect spikes: peaks above -20 mV, separated by at least 5 ms
min_samples_apart = int(0.005 * fs)   # 5 ms in samples
peak_indices, properties = signal.find_peaks(
    voltage,
    height=-20,                  # ignore peaks below -20 mV
    distance=min_samples_apart   # minimum spacing between detected spikes
)

print(f"Number of spikes detected: {len(peak_indices)}")
print(f"Spike times (ms): {t[peak_indices] * 1000}")

plt.figure(figsize=(11, 4))
plt.plot(t * 1000, voltage, linewidth=0.7, color="steelblue", label="Voltage")
plt.plot(t[peak_indices] * 1000, voltage[peak_indices], "rv",
         markersize=12, label="Detected spikes")
plt.axhline(-20, color="orange", linestyle="--", label="Detection threshold (-20 mV)")
plt.xlabel("Time (ms)")
plt.ylabel("Membrane potential (mV)")
plt.title("Spike detection in a simulated voltage recording")
plt.legend()
plt.tight_layout()
plt.show()


---
> ## Going deeper (optional on a first pass)
>
> **Spectrogram (time-frequency analysis).** A power spectrum averages over
> the whole recording. A spectrogram shows how the frequency content changes
> over time. Use `scipy.signal.spectrogram(data, fs)` or `plt.specgram()`.
> Invaluable for brain signals that switch between rhythms (e.g., alpha during
> rest, gamma during attention).
>
> **Welch's method.** A single FFT of a noisy signal is itself noisy. Welch's
> method (`scipy.signal.welch()`) divides the signal into overlapping segments,
> FFTs each one, and averages the power spectra. The result is a much smoother
> estimate of the true power spectrum. Standard practice in neuroscience.
>
> **Convolution.** Filtering is mathematically a convolution: sliding a
> "template" (the filter kernel) over the signal and computing a weighted local
> average at each point. `np.convolve()` or `scipy.signal.convolve()`.
> Understanding convolution also underpins how neural networks (CNNs) process
> images.
>
> **Zero-phase filtering.** `sosfilt()` introduces a small time delay (phase
> shift). To eliminate even this delay, use `sosfiltfilt()` which runs the
> filter forwards then backwards, cancelling the phase shift. Important when
> event timing matters.


In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

# Going deeper: Welch power spectrum (smoother than raw FFT)
rng = np.random.default_rng(5)
fs = 1000.0
duration = 10.0   # longer signal for Welch to work well
t = np.arange(0, duration, 1.0 / fs)
lfp_long = 50.0 * np.sin(2 * np.pi * 10 * t) + rng.normal(0, 30.0, size=t.size)

# Welch method: averages over overlapping windows for a cleaner spectrum
freqs_w, power_w = signal.welch(lfp_long, fs=fs, nperseg=512)
# nperseg = samples per segment (longer = finer frequency resolution)

plt.figure(figsize=(8, 4))
plt.semilogy(freqs_w, power_w, color="darkorange", linewidth=2)
plt.axvline(10, color="red", linestyle="--", label="10 Hz")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power spectral density (uV^2 / Hz)")
plt.title("Welch power spectrum (smoother than raw FFT)")
plt.xlim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()


## Common questions and confusions

**"What does 'order' mean in the Butterworth filter?"** The order controls
how steeply the filter transitions from pass to stop. Order 1 is a gentle
slope; order 8 is very sharp. Higher order looks cleaner but can cause
"ringing" (oscillation near sharp edges in the data). Order 4 is a safe
default for most neuroscience applications.

**"Why does FFT output have negative frequencies?"** The FFT decomposes a
real signal into complex exponentials, which naturally come in positive/negative
pairs. For a real (non-complex) signal the two halves mirror each other, so
you only need the positive half. `rfft()` skips the redundant half automatically.

**"My filtered signal looks delayed compared to the original. Why?"** A causal
filter (like `sosfilt`) introduces a phase lag. Use `sosfiltfilt()` for zero
phase delay if timing is critical.

**"Why does `find_peaks` miss some spikes?"** The `height` or `distance`
parameters may be too strict or too loose. Always plot the result with detected
peaks overlaid so you can see what is being missed or falsely detected.


## Your exercises

1. Generate a signal that is a sum of two sine waves: 5 Hz at amplitude 1.0 and
   50 Hz at amplitude 0.3. Compute and plot the power spectrum. Confirm you see
   two peaks at 5 Hz and 50 Hz.

2. Apply a low-pass filter at 20 Hz to the signal from exercise 1 (use fs=500 Hz).
   Plot the filtered signal alongside the original. Has the 50 Hz component been
   removed?

3. Apply a band-pass filter between 3 Hz and 7 Hz to isolate just the 5 Hz
   component. Use `btype="bandpass"` and pass `[3.0, 7.0]` as the cutoff
   (a list of two frequencies). Plot the result.

4. Simulate a 500 ms voltage trace (fs=10000 Hz) with baseline noise around
   -65 mV. Insert 3 spikes manually (add a step of +60 mV for 1 sample at
   times 50 ms, 200 ms, 400 ms). Use `find_peaks` to detect them. Experiment
   with the `height` parameter.

5. Generate a 10-second LFP containing a 10 Hz rhythm. Use `scipy.signal.welch()`
   to compute the power spectrum. Try different `nperseg` values (128, 512, 2048)
   and describe how frequency resolution and smoothness change.

6. *(Stretch.)* Generate a signal that switches from a 10 Hz rhythm (first 3 s)
   to a 40 Hz rhythm (last 3 s). Compute `scipy.signal.spectrogram()` and plot
   it. Can you see the transition in time?


In [ ]:
# Exercise 1
# your code here

# Exercise 2
# your code here

# Exercise 3
# your code here

# Exercise 4
# your code here

# Exercise 5
# your code here

# Exercise 6 (stretch)
# your code here


## The irreducible core

1. A **low-pass filter** removes fast noise and keeps slow structure; a
   **band-pass filter** isolates a specific frequency range.
2. Design a filter with `butter(order, cutoff, btype, fs, output="sos")`;
   apply it with `sosfilt(sos, data)`.
3. The **Fourier transform** decomposes a signal into its component frequencies;
   `fft()` + `fftfreq()` give power vs frequency (the power spectrum).
4. `find_peaks()` detects local maxima; use `height` and `distance` to set
   thresholds for spike detection.
5. Key neuroscience terms: **LFP** = the slow electrical hum of a brain region;
   **power spectrum** = energy at each frequency; **Hz** = cycles per second.

**You have got it when:** given a noisy LFP recording with a hidden 10 Hz
rhythm, you can filter it, compute its power spectrum, and show the 10 Hz peak.


---
# Item 4 — Integration and ODEs with `scipy.integrate`

> **By the end of this you will be able to:** compute the area under a curve
> numerically using `quad`, explain what an ODE is in plain English, and use
> `solve_ivp` to simulate systems that evolve over time, including a leaky
> integrate-and-fire neuron. Everything below the core sections is optional on
> a first pass.


## Numerical integration: area under a curve

### Everyday picture 1: the area under a drug concentration curve

A patient receives a single dose of antibiotic. The blood concentration rises,
peaks, then gradually falls over 24 hours. The total exposure of the body to the
drug is the area under that concentration-vs-time curve. Doctors call this the
AUC (area under the curve) and use it to set dosing intervals. Computing this
area is numerical integration.

### Everyday picture 2: distance from velocity

A car's speedometer shows 60 km/h. If you want to know how far it has travelled
in 2 hours, you multiply: 60 x 2 = 120 km. But if the speed changes every
minute, you cannot simply multiply; you have to add up the tiny distance covered
each second. That is integration: summing infinitely many infinitely thin
rectangles under the velocity curve to get total distance.

`scipy.integrate.quad(f, a, b)` does exactly this: numerically computes the
definite integral (area) of function `f` from `a` to `b`. It returns two values:
the estimate and the estimated numerical error.


In [ ]:
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------
# Example 1: area under a drug concentration curve
# -----------------------------------------------------------------------
# Simplified pharmacokinetics: concentration rises then decays exponentially.
# C(t) = C_peak * t * exp(-k * t)   where k controls the elimination rate.

C_peak = 10.0   # peak concentration (arbitrary units)
k = 0.5         # elimination rate constant (1/hour)

def concentration(t):
    return C_peak * t * np.exp(-k * t)

# Integrate from t=0 to t=24 hours: total drug exposure (AUC)
AUC, error_estimate = integrate.quad(concentration, 0, 24)
print(f"AUC (total drug exposure, 0-24 h): {AUC:.3f} units.hours")
print(f"Numerical error estimate:          {error_estimate:.2e}")

# Visualise the curve and the area
t_plot = np.linspace(0, 24, 300)
C_plot = concentration(t_plot)
plt.figure(figsize=(8, 4))
plt.fill_between(t_plot, C_plot, alpha=0.3, color="steelblue", label="AUC (shaded)")
plt.plot(t_plot, C_plot, color="steelblue", linewidth=2)
plt.xlabel("Time (hours)")
plt.ylabel("Drug concentration (units)")
plt.title(f"Drug concentration over 24 h  |  AUC = {AUC:.2f}")
plt.legend()
plt.tight_layout()
plt.show()

# -----------------------------------------------------------------------
# Example 2: integrate a firing rate function over time
# -----------------------------------------------------------------------
# A neuron's firing rate (Hz) changes during a stimulus epoch.
# The total number of spikes is the integral of the rate over time.

def firing_rate(t):
    # Gaussian bump: rate peaks at t=0.5 s, width 0.1 s
    return 80.0 * np.exp(-0.5 * ((t - 0.5) / 0.1) ** 2)

total_spikes, _ = integrate.quad(firing_rate, 0.0, 1.0)
print(f"\nTotal spike count over 1-second window: {total_spikes:.1f} spikes")


## Ordinary Differential Equations (ODEs): rules for how things change

### What an ODE is in plain English

An **ODE (Ordinary Differential Equation)** is a rule that says:
"given the current state of the system, here is how fast it is changing RIGHT NOW."

### Everyday picture 1: a leaky bucket

You fill a bucket with water and then make a hole in the bottom. How the water
level changes depends on the current level -- high level means high pressure
means fast outflow. The rule is: "rate of decrease = proportional to current
level." That is an ODE. You cannot compute tomorrow's level directly; you have
to follow the rule step by step.

### Everyday picture 2: a ball thrown in the air

"The ball's velocity changes because gravity pulls it down at 9.8 m/s per
second." That is an ODE: `dv/dt = -9.8`. Given starting height and velocity,
you can compute the whole trajectory by following this rule forward in time.

### `solve_ivp`: solving an ODE step by step

`scipy.integrate.solve_ivp(fun, t_span, y0)` takes:
- **`fun`** -- the rule: a function `f(t, y)` that returns `dy/dt` (the current
  rate of change, given the current time `t` and state `y`).
- **`t_span`** -- `(t_start, t_end)`: the time interval to simulate.
- **`y0`** -- the initial state (starting values).
- **`t_eval`** -- (optional) the times at which you want the output stored.

It returns a result object. Use `.t` for the time points and `.y` for the
state at each time.


In [ ]:
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------
# Exponential decay ODE: post-synaptic potential
# -----------------------------------------------------------------------
# A synapse (the junction between two neurons) receives a signal and briefly
# increases its conductance. The resulting voltage change (post-synaptic
# potential, PSP) then decays exponentially back to baseline.
# ODE: dV/dt = -(V - V_rest) / tau
# This says: "the voltage decays towards resting at a rate proportional to
# how far it currently is from resting." tau (tau) is the time constant (ms).

V_rest = -70.0    # resting membrane potential (mV)
tau = 10.0        # time constant (ms)

def psp_ode(t, y):
    V = y[0]                     # current voltage (the state)
    dVdt = -(V - V_rest) / tau   # how fast voltage changes right now
    return [dVdt]

t_span = (0.0, 80.0)             # simulate 0 to 80 ms
y0 = [-50.0]                     # start at -50 mV (displaced from rest)
t_eval = np.linspace(0, 80, 500) # times at which we want the solution stored

sol = integrate.solve_ivp(psp_ode, t_span, y0, t_eval=t_eval)
# sol.t : time array
# sol.y : 2D array, shape (n_variables, n_timepoints); row 0 = voltage

plt.figure(figsize=(8, 4))
plt.plot(sol.t, sol.y[0], color="steelblue", linewidth=2)
plt.axhline(V_rest, linestyle="--", color="gray", label=f"Resting potential ({V_rest} mV)")
plt.xlabel("Time (ms)")
plt.ylabel("Membrane potential (mV)")
plt.title("Exponential decay of a post-synaptic potential (ODE simulation)")
plt.legend()
plt.tight_layout()
plt.show()

# Verify: analytical solution for comparison
V_analytical = V_rest + (-50.0 - V_rest) * np.exp(-sol.t / tau)
max_error = np.max(np.abs(sol.y[0] - V_analytical))
print(f"Maximum error vs analytical solution: {max_error:.2e} mV  (should be tiny)")


## The Leaky Integrate-and-Fire (LIF) neuron: the canonical simple neuron model

The **Leaky Integrate-and-Fire (LIF) model** is the most widely used simple
model of a neuron. It captures the two essential behaviours of real neurons:

1. **Integration:** the membrane potential slowly rises as input current flows in
   (like water filling a bucket).
2. **Leaking:** the membrane potential also constantly leaks back towards the
   resting value (a hole in the bucket).
3. **Firing (the "fire" part):** when the potential crosses a threshold (typically
   around -50 mV), the neuron fires a spike (action potential). The voltage is then
   reset to a reset value and the process restarts.

The ODE describing the LIF is:

```
tau_m * dV/dt = -(V - V_rest) + R * I(t)
```

Where:
- `tau_m` = membrane time constant (how quickly the cell integrates current)
- `V` = membrane potential (the state variable)
- `V_rest` = resting potential
- `R` = membrane resistance
- `I(t)` = input current (the "drive")

We simulate this with `solve_ivp`, manually detecting the threshold crossing
and resetting voltage after each spike.


In [ ]:
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------
# Leaky Integrate-and-Fire (LIF) neuron simulation with solve_ivp
# -----------------------------------------------------------------------

# Biophysical parameters
tau_m = 20.0        # membrane time constant (ms): how fast voltage integrates
V_rest = -70.0      # resting potential (mV)
V_thresh = -50.0    # spike threshold (mV): fire when voltage crosses this
V_reset = -75.0     # reset potential after a spike (briefly more negative, refractory)
R_m = 10.0          # membrane resistance (MOhm, megaohms)
I_input = 2.5       # constant input current (nA, nanoamps)
# This current is enough to drive the neuron to fire repeatedly.

# We will manually detect spikes and reset.
# solve_ivp with events is one approach; here we use a simple step-by-step loop
# for clarity, then compare with an ODE approach.

dt = 0.1            # time step (ms)
T = 300.0           # total simulation time (ms)
t_arr = np.arange(0, T, dt)
V = np.zeros_like(t_arr)
V[0] = V_rest

spike_times = []

for i in range(1, len(t_arr)):
    # LIF ODE: tau_m * dV/dt = -(V - V_rest) + R_m * I
    dVdt = (-(V[i-1] - V_rest) + R_m * I_input) / tau_m
    V[i] = V[i-1] + dVdt * dt

    # Spike detection and reset
    if V[i] >= V_thresh:
        spike_times.append(t_arr[i])  # record the spike time
        V[i] = V_reset                # reset voltage

spike_times = np.array(spike_times)
if len(spike_times) > 1:
    isis = np.diff(spike_times)       # inter-spike intervals (ms)
    mean_rate = 1000.0 / np.mean(isis)  # convert ms interval to Hz firing rate

print(f"Number of spikes: {len(spike_times)}")
if len(spike_times) > 1:
    print(f"Mean ISI: {np.mean(isis):.1f} ms")
    print(f"Mean firing rate: {mean_rate:.1f} Hz")

# Plot membrane potential over time
plt.figure(figsize=(12, 4))
plt.plot(t_arr, V, color="steelblue", linewidth=1.0)
plt.axhline(V_thresh, color="red", linestyle="--", linewidth=1, label=f"Threshold ({V_thresh} mV)")
plt.axhline(V_rest, color="gray", linestyle=":", linewidth=1, label=f"Rest ({V_rest} mV)")
for st in spike_times:
    plt.axvline(st, color="orange", linewidth=0.7, alpha=0.6)
plt.xlabel("Time (ms)")
plt.ylabel("Membrane potential (mV)")
plt.title("Leaky Integrate-and-Fire neuron: membrane potential with spikes")
plt.legend()
plt.tight_layout()
plt.show()

# Raster plot: just the spike times
plt.figure(figsize=(12, 1.5))
plt.eventplot([spike_times], colors="black", linewidths=1.5)
plt.xlabel("Time (ms)")
plt.yticks([])
plt.title("Spike raster (each bar = one action potential)")
plt.tight_layout()
plt.show()


---
> ## Going deeper (optional on a first pass)
>
> **Stiff ODEs.** Some ODEs have components that change at vastly different
> speeds (e.g., fast gating variables and slow membrane potential). These are
> "stiff" and require implicit solvers. `solve_ivp` with `method="Radau"` or
> `method="BDF"` handles stiff problems. The Hodgkin-Huxley model below is
> stiff.
>
> **The Hodgkin-Huxley model.** The LIF is a simplification. The Hodgkin-Huxley
> model (1952, Nobel Prize 1963) describes the actual ionic channels responsible
> for the action potential using four coupled ODEs for membrane potential and
> three "gating variables" (m, n, h) controlling sodium and potassium currents.
> It is far more realistic and produces authentic spike shapes, but is much
> harder to simulate. Solving it is a standard exercise in computational
> neuroscience.
>
> **`scipy.integrate.odeint` (older API).** The older function `odeint(f, y0, t)`
> is still commonly seen in neuroscience code. It works similarly to `solve_ivp`
> but with a different argument order (`f(y, t)` instead of `f(t, y)`). Prefer
> `solve_ivp` for new code; use `odeint` when reading others' scripts.
>
> **Events in `solve_ivp`.** You can pass an `events` argument: a function that
> returns zero at the moment of interest (e.g., threshold crossing). `solve_ivp`
> will find those crossings exactly and can stop or log them. This is cleaner
> than the manual loop above for precise spike timing.


In [ ]:
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt

# Going deeper: use solve_ivp with an event to detect threshold crossings
# This is the cleaner ODE approach (vs the manual Euler loop above)

tau_m = 20.0
V_rest = -70.0
V_thresh = -50.0
R_m = 10.0
I_input = 2.5

def lif_ode(t, y):
    V = y[0]
    dVdt = (-(V - V_rest) + R_m * I_input) / tau_m
    return [dVdt]

def threshold_event(t, y):
    return y[0] - V_thresh   # zero when V reaches threshold

threshold_event.terminal = True    # stop integration when event fires
threshold_event.direction = 1      # only detect upward crossings

# Simulate one inter-spike interval
sol = integrate.solve_ivp(lif_ode, (0, 500), [V_rest],
                          events=threshold_event, t_eval=np.linspace(0, 500, 5000))

print("Spike occurred at t =", sol.t_events[0], "ms")

plt.figure(figsize=(8, 4))
plt.plot(sol.t, sol.y[0], color="steelblue", linewidth=2)
plt.axhline(V_thresh, color="red", linestyle="--", label="Threshold")
plt.xlabel("Time (ms)")
plt.ylabel("Membrane potential (mV)")
plt.title("solve_ivp with threshold event: stops exactly at spike")
plt.legend()
plt.tight_layout()
plt.show()


## Common questions and confusions

**"What is the difference between `quad` and `solve_ivp`?"** `quad` computes
a single number: the area under a given function from a to b. `solve_ivp`
simulates a system forward in time, step by step, using an ODE rule. They
use related mathematics but serve different purposes.

**"What is `y0` and why is it a list?"** `y0` is the initial state of the
system at time zero. It is a list (or array) because a system can have multiple
state variables (e.g., the Hodgkin-Huxley model has four: V, m, n, h).
Even if you only have one variable (membrane voltage), you still pass it as a
list `[V_start]` and access it as `sol.y[0]`.

**"Why does the ODE function take `(t, y)` but `t` is not used?"** Many ODEs
are "autonomous" (the rule does not explicitly depend on time, only on the
current state). You still have to include `t` as the first argument because
`solve_ivp` will call your function with `f(t, y)` regardless. Just ignore
`t` inside the function if you don't need it.

**"My LIF simulation never fires. Why?"** Either the input current is below
the rheobase (the minimum current needed to make the neuron fire), or the
time step `dt` is too coarse to catch the threshold crossing. Try increasing
`I_input` or decreasing `dt`.


## Your exercises

1. Use `scipy.integrate.quad` to compute the integral of `sin(x)` from `0`
   to `pi`. What should the exact answer be? Compare with `quad`'s result.

2. Define an ODE for exponential growth: `dy/dt = 0.1 * y`, `y(0) = 1`.
   Use `solve_ivp` to simulate from `t=0` to `t=30`. Plot the result.
   Overlay the analytical solution `y = exp(0.1 * t)`. Do they match?

3. Simulate the post-synaptic potential decay ODE from the worked example
   but change `tau` to 5 ms, then 50 ms. Plot both. How does `tau` control
   the speed of decay?

4. Run the LIF neuron with `I_input = 1.5` (below threshold), `I_input = 2.5`
   (moderate), and `I_input = 5.0` (strong). Plot all three. How does input
   current affect firing rate?

5. Compute the AUC (area under the curve) of the firing rate function
   `rate(t) = 50 * exp(-t / 0.2)` (a decaying burst) from `t=0` to `t=2`
   seconds. This gives the total expected spike count.

6. *(Stretch.)* Model a leaky bucket: water flows in at a constant rate `Q_in`
   and leaks out proportional to the current height `h` with rate constant `k`.
   ODE: `dh/dt = Q_in - k * h`. Simulate for 60 seconds. What is the
   steady-state height (where inflow = outflow)? Verify with `solve_ivp`.


In [ ]:
# Exercise 1
# your code here

# Exercise 2
# your code here

# Exercise 3
# your code here

# Exercise 4
# your code here

# Exercise 5
# your code here

# Exercise 6 (stretch)
# your code here


## The irreducible core

1. `scipy.integrate.quad(f, a, b)` computes the area under `f` between `a`
   and `b`; returns `(value, error_estimate)`.
2. An **ODE** is a rule: "given the current state, here is the rate of change
   right now." It is solved by following the rule forward in time, step by step.
3. `solve_ivp(f, t_span, y0)` requires: the ODE function `f(t, y)`, the time
   range `t_span`, and the initial state `y0` (always a list).
4. The **LIF neuron** integrates input current, leaks towards rest, and fires
   (resets) when it crosses a threshold. It is the starting model for all of
   computational neuroscience.
5. The time constant `tau` controls speed: large `tau` = slow integration (lazy
   neuron), small `tau` = fast response.

**You have got it when:** you can write the LIF ODE as a Python function,
pass it to `solve_ivp`, detect the spikes, and compute the mean firing rate.


---
# Solutions -- try first!

Work through every exercise yourself before reading these. The struggle is
where the learning happens.


## Item 1 Solutions -- scipy.stats

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)

# --- Exercise 1 ---
data = rng.normal(50, 10, 100)
d = stats.describe(data)
print("Exercise 1:")
print(f"  Mean:     {d.mean:.2f}  (expected ~50)")
print(f"  Variance: {d.variance:.2f}  (expected ~100 = 10^2)")
print(f"  Skewness: {d.skewness:.3f}  (expected ~0, symmetric)")
print(f"  Kurtosis: {d.kurtosis:.3f}  (expected ~0 for normal)")


In [ ]:
import numpy as np
from scipy import stats

# --- Exercise 2 ---
# Resting membrane potential: normal, mean -70 mV, SD 5 mV.
# P(V < -80) = CDF at -80.
p_below_minus80 = stats.norm.cdf(-80, loc=-70, scale=5)
print(f"Exercise 2: P(V < -80 mV) = {p_below_minus80:.4f}  ({p_below_minus80*100:.2f}%)")
# -80 is 2 standard deviations below the mean, so ~2.3% -- relatively rare.


In [ ]:
from scipy import stats

# --- Exercise 3 ---
v99 = stats.norm.ppf(0.99, loc=-70, scale=5)
print(f"Exercise 3: 99th percentile of membrane potential = {v99:.2f} mV")
# About -58.4 mV: only 1% of readings exceed this (very depolarised)


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(1)

# --- Exercise 4 ---
a = rng.normal(30, 6, 25)
b = rng.normal(33, 6, 25)
t, p = stats.ttest_ind(a, b)
print(f"Exercise 4 (n=25): mean_a={a.mean():.1f}  mean_b={b.mean():.1f}  p={p:.4f}")
print("  Significant?" , "Yes" if p < 0.05 else "No")
print("  A 3 Hz difference with n=25 and SD=6 may not reach significance.")

# --- Exercise 5 ---
a2 = rng.normal(30, 6, 200)
b2 = rng.normal(33, 6, 200)
t2, p2 = stats.ttest_ind(a2, b2)
print(f"\nExercise 5 (n=200): mean_a={a2.mean():.1f}  mean_b={b2.mean():.1f}  p={p2:.4f}")
print("  Significant?" , "Yes" if p2 < 0.05 else "No")
print("  Larger sample -> smaller standard error -> easier to detect real differences.")


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(99)

# --- Exercise 6 (stretch) ---
normal_data = rng.normal(20, 5, 50)
exp_data = rng.exponential(20, 50)

_, p_sw_norm = stats.shapiro(normal_data)
_, p_sw_exp = stats.shapiro(exp_data)
print(f"Shapiro-Wilk p (normal data): {p_sw_norm:.3f}  -> {'Looks normal' if p_sw_norm > 0.05 else 'NOT normal'}")
print(f"Shapiro-Wilk p (exp data):   {p_sw_exp:.4f}  -> {'Looks normal' if p_sw_exp > 0.05 else 'NOT normal'}")

_, p_t = stats.ttest_ind(normal_data, exp_data)
_, p_mw = stats.mannwhitneyu(normal_data, exp_data, alternative="two-sided")
print(f"\nt-test p:          {p_t:.4f}")
print(f"Mann-Whitney U p:  {p_mw:.4f}")
print("When exp_data is non-normal, Mann-Whitney is more reliable.")
print("Both may detect a real difference here, but MWU makes fewer assumptions.")


## Item 2 Solutions -- scipy.optimize

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

rng = np.random.default_rng(0)

# --- Exercise 1 ---
t = np.linspace(0, 5, 100)
y = 3.0 * np.exp(-t / 1.5) + rng.normal(0, 0.2, 100)

def exp_decay(t, A, tau):
    return A * np.exp(-t / tau)

popt, pcov = curve_fit(exp_decay, t, y, p0=[2.5, 1.0])
perr = np.sqrt(np.diag(pcov))
print("Exercise 1:")
print(f"  Amplitude A = {popt[0]:.3f} +/- {perr[0]:.3f}  (true: 3.0)")
print(f"  Tau         = {popt[1]:.3f} +/- {perr[1]:.3f}  (true: 1.5)")


In [ ]:
import numpy as np
from scipy.optimize import curve_fit

rng = np.random.default_rng(1)

# --- Exercise 2 ---
x = np.linspace(0, 10, 80)
y_true = 2.5 * x + 1.0
y_noisy = y_true + rng.normal(0, 1.0, 80)

def linear(x, m, c):
    return m * x + c

popt, _ = curve_fit(linear, x, y_noisy)
poly = np.polyfit(x, y_noisy, 1)   # numpy polynomial fit as comparison

print("Exercise 2:")
print(f"  curve_fit: m={popt[0]:.3f}  c={popt[1]:.3f}")
print(f"  polyfit:   m={poly[0]:.3f}  c={poly[1]:.3f}  (should agree closely)")


In [ ]:
import numpy as np
from scipy.optimize import curve_fit

# --- Exercise 3 ---
dose = np.array([0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0])
response = np.array([0.02, 0.05, 0.12, 0.28, 0.50, 0.75, 0.92, 0.98])

def logistic(x, ec50, slope):
    return 1.0 / (1.0 + (ec50 / x) ** slope)

popt, _ = curve_fit(logistic, dose, response, p0=[1.0, 1.5],
                    bounds=([0, 0.1], [100, 10]))
ec50_fit, slope_fit = popt
print(f"Exercise 3: EC50 = {ec50_fit:.3f}  slope = {slope_fit:.2f}")
print(f"  At dose {ec50_fit:.3f}, response = {logistic(ec50_fit, *popt):.2f}  (should be 0.5)")


In [ ]:
import numpy as np
from scipy.optimize import curve_fit

rng = np.random.default_rng(0)

# --- Exercise 4 ---
V0_true, V_rest_true, tau_true = -50.0, -70.0, 20.0
t = np.linspace(0, 100, 200)
V_data = V_rest_true + (V0_true - V_rest_true) * np.exp(-t / tau_true) + rng.normal(0, 0.8, 200)

def membrane_decay(t, V0, V_rest, tau):
    return V_rest + (V0 - V_rest) * np.exp(-t / tau)

# Deliberately offset V_rest guess
popt, pcov = curve_fit(membrane_decay, t, V_data, p0=[-55.0, -60.0, 15.0])
perr = np.sqrt(np.diag(pcov))
print("Exercise 4 (bad V_rest guess):")
print(f"  V0    = {popt[0]:.2f} +/- {perr[0]:.2f}  (true: {V0_true})")
print(f"  V_rest= {popt[1]:.2f} +/- {perr[1]:.2f}  (true: {V_rest_true})")
print(f"  tau   = {popt[2]:.2f} +/- {perr[2]:.2f}  (true: {tau_true})")
print("Fit typically converges even with off guesses for this smooth function.")


In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)

# --- Exercise 5 (stretch) ---
t = np.linspace(0.01, 5, 200)
y = 2 * np.exp(-t / 0.5) + 5 * np.exp(-t / 5.0) + rng.normal(0, 0.15, 200)

def single_exp(t, A, tau):
    return A * np.exp(-t / tau)

def double_exp(t, A1, tau1, A2, tau2):
    return A1 * np.exp(-t / tau1) + A2 * np.exp(-t / tau2)

popt1, _ = curve_fit(single_exp, t, y, p0=[5, 2])
popt2, _ = curve_fit(double_exp, t, y, p0=[2, 0.5, 5, 5])

res1 = y - single_exp(t, *popt1)
res2 = y - double_exp(t, *popt2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t, res1, ".", markersize=3, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Single exponential residuals (systematic bend = wrong model)")
axes[0].set_xlabel("t")
axes[1].plot(t, res2, ".", markersize=3, color="green")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Double exponential residuals (random scatter = good model)")
axes[1].set_xlabel("t")
plt.tight_layout()
plt.show()
print("Single exp SS residuals:", np.sum(res1**2).round(3))
print("Double exp SS residuals:", np.sum(res2**2).round(3))


## Item 3 Solutions -- scipy.signal

In [ ]:
import numpy as np
from scipy import signal, fft as scipy_fft
import matplotlib.pyplot as plt

rng = np.random.default_rng(10)
fs = 500.0

# --- Exercise 1 ---
t = np.arange(0, 2, 1/fs)
sig = np.sin(2*np.pi*5*t) + 0.3*np.sin(2*np.pi*50*t)

N = len(sig)
fft_vals = scipy_fft.fft(sig)
freqs = scipy_fft.fftfreq(N, d=1.0/fs)
power = np.abs(fft_vals)**2 / N
pos = freqs >= 0

plt.figure(figsize=(8,3))
plt.plot(freqs[pos], power[pos])
plt.axvline(5, color="red", linestyle="--", label="5 Hz")
plt.axvline(50, color="orange", linestyle="--", label="50 Hz")
plt.xlim(0, 100); plt.xlabel("Frequency (Hz)"); plt.ylabel("Power")
plt.title("Exercise 1: power spectrum with 5 Hz and 50 Hz peaks")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

fs = 500.0
t = np.arange(0, 2, 1/fs)
sig = np.sin(2*np.pi*5*t) + 0.3*np.sin(2*np.pi*50*t)

# --- Exercise 2 ---
sos = signal.butter(4, 20.0, btype="low", fs=fs, output="sos")
filtered = signal.sosfilt(sos, sig)

plt.figure(figsize=(9,4))
plt.plot(t[:200], sig[:200], label="Original", color="lightblue", linewidth=0.8)
plt.plot(t[:200], filtered[:200], label="Low-pass filtered (20 Hz)", color="navy", linewidth=2)
plt.xlabel("Time (s)"); plt.ylabel("Amplitude")
plt.title("Exercise 2: low-pass filter removes 50 Hz component")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

fs = 500.0
t = np.arange(0, 2, 1/fs)
sig = np.sin(2*np.pi*5*t) + 0.3*np.sin(2*np.pi*50*t)

# --- Exercise 3 ---
sos_bp = signal.butter(4, [3.0, 7.0], btype="bandpass", fs=fs, output="sos")
bp_filtered = signal.sosfilt(sos_bp, sig)

plt.figure(figsize=(9,4))
plt.plot(t[:300], sig[:300], label="Original", color="lightblue", linewidth=0.8)
plt.plot(t[:300], bp_filtered[:300], label="Band-pass 3-7 Hz", color="darkgreen", linewidth=2)
plt.xlabel("Time (s)"); plt.ylabel("Amplitude")
plt.title("Exercise 3: band-pass filter isolates 5 Hz component")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)
fs = 10000.0
t = np.arange(0, 0.5, 1/fs)
voltage = rng.normal(-65.0, 1.5, size=t.size)

# --- Exercise 4 ---
for spike_ms in [50, 200, 400]:
    idx = int(spike_ms * 1e-3 * fs)
    if idx < len(voltage):
        voltage[idx] += 60.0  # brief +60 mV spike

peak_idx, _ = signal.find_peaks(voltage, height=-20.0, distance=int(0.01*fs))
print(f"Exercise 4: detected {len(peak_idx)} spikes at times (ms):", t[peak_idx]*1000)

plt.figure(figsize=(10,3))
plt.plot(t*1000, voltage, linewidth=0.5, color="steelblue")
plt.plot(t[peak_idx]*1000, voltage[peak_idx], "rv", markersize=10, label="Detected")
plt.axhline(-20, color="orange", linestyle="--", label="Threshold")
plt.xlabel("Time (ms)"); plt.ylabel("mV")
plt.title("Exercise 4: spike detection"); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)
fs = 1000.0
t = np.arange(0, 10, 1/fs)
lfp_long = 50*np.sin(2*np.pi*10*t) + rng.normal(0, 30, t.size)

# --- Exercise 5 ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, nperseg, label in zip(axes, [128, 512, 2048], ["128 (coarse)", "512 (medium)", "2048 (fine)"]):
    f, p = signal.welch(lfp_long, fs=fs, nperseg=nperseg)
    ax.semilogy(f, p)
    ax.axvline(10, color="red", linestyle="--")
    ax.set_xlim(0, 80); ax.set_xlabel("Hz"); ax.set_ylabel("PSD")
    ax.set_title(f"nperseg={label}")
plt.suptitle("Exercise 5: Welch PSD with different nperseg values")
plt.tight_layout(); plt.show()
print("Larger nperseg: finer frequency resolution, smoother (more averaging)")
print("Smaller nperseg: coarser resolution, noisier spectrum")


In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
fs = 1000.0

# --- Exercise 6 (stretch) ---
t1 = np.arange(0, 3, 1/fs)
t2 = np.arange(3, 6, 1/fs)
seg1 = 40*np.sin(2*np.pi*10*t1) + rng.normal(0, 10, t1.size)  # 10 Hz
seg2 = 30*np.sin(2*np.pi*40*t2) + rng.normal(0, 10, t2.size)  # 40 Hz
full_signal = np.concatenate([seg1, seg2])

f, t_spec, Sxx = signal.spectrogram(full_signal, fs=fs, nperseg=256)
plt.figure(figsize=(10, 4))
plt.pcolormesh(t_spec, f[:80], Sxx[:80], shading="gouraud", cmap="viridis")
plt.ylabel("Frequency (Hz)")
plt.xlabel("Time (s)")
plt.title("Exercise 6: spectrogram showing switch from 10 Hz to 40 Hz at t=3 s")
plt.colorbar(label="Power")
plt.tight_layout()
plt.show()


## Item 4 Solutions -- scipy.integrate

In [ ]:
import numpy as np
from scipy import integrate

# --- Exercise 1 ---
result, err = integrate.quad(np.sin, 0, np.pi)
print(f"Exercise 1: integral of sin(x) from 0 to pi = {result:.6f}")
print(f"  Exact answer = 2.0  |  Error estimate: {err:.2e}")


In [ ]:
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt

# --- Exercise 2 ---
def exp_growth(t, y):
    return [0.1 * y[0]]   # dy/dt = 0.1 * y

t_eval = np.linspace(0, 30, 300)
sol = integrate.solve_ivp(exp_growth, (0, 30), [1.0], t_eval=t_eval)

y_analytical = np.exp(0.1 * t_eval)

plt.figure(figsize=(8,4))
plt.plot(sol.t, sol.y[0], label="solve_ivp", linewidth=2)
plt.plot(t_eval, y_analytical, "--", label="Analytical exp(0.1t)", linewidth=1.5)
plt.xlabel("t"); plt.ylabel("y")
plt.title("Exercise 2: exponential growth ODE vs analytical solution")
plt.legend(); plt.tight_layout(); plt.show()
print(f"Max error: {np.max(np.abs(sol.y[0] - y_analytical)):.2e}  (should be tiny)")


In [ ]:
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt

# --- Exercise 3 ---
V_rest = -70.0
y0 = [-50.0]
t_eval = np.linspace(0, 100, 500)

plt.figure(figsize=(9, 4))
for tau, col in [(5.0, "red"), (20.0, "steelblue"), (50.0, "green")]:
    def psp(t, y, tau=tau):
        return [-(y[0] - V_rest) / tau]
    sol = integrate.solve_ivp(psp, (0, 100), y0, t_eval=t_eval)
    plt.plot(sol.t, sol.y[0], color=col, linewidth=2, label=f"tau={tau} ms")

plt.axhline(V_rest, linestyle="--", color="gray")
plt.xlabel("Time (ms)"); plt.ylabel("Membrane potential (mV)")
plt.title("Exercise 3: larger tau = slower decay to resting potential")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Exercise 4 ---
tau_m, V_rest, V_thresh, V_reset, R_m = 20.0, -70.0, -50.0, -75.0, 10.0
dt = 0.1
T = 500.0
t_arr = np.arange(0, T, dt)

plt.figure(figsize=(12, 8))
for i, I_input in enumerate([1.5, 2.5, 5.0]):
    V = np.zeros_like(t_arr)
    V[0] = V_rest
    spikes = 0
    for j in range(1, len(t_arr)):
        dVdt = (-(V[j-1] - V_rest) + R_m * I_input) / tau_m
        V[j] = V[j-1] + dVdt * dt
        if V[j] >= V_thresh:
            spikes += 1
            V[j] = V_reset
    rate = spikes / (T / 1000.0)
    plt.subplot(3, 1, i+1)
    plt.plot(t_arr, V, linewidth=0.8, color="steelblue")
    plt.axhline(V_thresh, color="red", linestyle="--")
    plt.title(f"I_input={I_input} nA  -> {spikes} spikes, ~{rate:.0f} Hz")
    plt.ylabel("mV")
plt.xlabel("Time (ms)")
plt.suptitle("Exercise 4: LIF neuron firing rate increases with input current")
plt.tight_layout(); plt.show()


In [ ]:
from scipy import integrate
import numpy as np

# --- Exercise 5 ---
def rate_burst(t):
    return 50 * np.exp(-t / 0.2)  # fast decaying burst (tau = 0.2 s)

total_spikes, err = integrate.quad(rate_burst, 0, 2)
print(f"Exercise 5: total expected spikes in 2 s burst = {total_spikes:.2f}")
print(f"  (= 50 * 0.2 * (1 - exp(-10)) analytically = {50*0.2*(1-np.exp(-10)):.2f})")


In [ ]:
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt

# --- Exercise 6 (stretch) ---
Q_in = 5.0    # inflow rate (litres/minute)
k = 0.1       # leak rate constant (1/minute)
# Steady state: dh/dt = 0 -> h_ss = Q_in / k
h_ss = Q_in / k
print(f"Exercise 6: steady-state height = Q_in / k = {h_ss:.1f} units")

def bucket_ode(t, y):
    h = y[0]
    dhdt = Q_in - k * h
    return [dhdt]

t_eval = np.linspace(0, 60, 600)
sol = integrate.solve_ivp(bucket_ode, (0, 60), [0.0], t_eval=t_eval)

plt.figure(figsize=(8, 4))
plt.plot(sol.t, sol.y[0], linewidth=2, color="steelblue", label="Water height")
plt.axhline(h_ss, color="red", linestyle="--", label=f"Steady state ({h_ss:.0f})")
plt.xlabel("Time (minutes)"); plt.ylabel("Height")
plt.title("Exercise 6: leaky bucket ODE -- approaches steady state")
plt.legend(); plt.tight_layout(); plt.show()
